# Risk Calculation Test - Neema Juma Case Study

This notebook tests the mastery-based risk calculation fixes.

## Setup

In [ ]:
import os
import sys
import django

# Setup Django environment
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'teachlink.settings')
django.setup()

from decimal import Decimal
from datetime import datetime, timedelta
from django.utils import timezone
from courses.models import Enrollment, Course, LessonCompletion
from assessments.models import Quiz, QuizAttempt
from analytics.services.risk_engine import RiskEngine
from analytics.services.alert_generator import AlertGenerator
from users.models import User

print("✓ Django environment initialized")

## 1. Current State - Neema Juma

Load Neema's enrollment and check current risk state.

In [ ]:
# Find Neema Juma's enrollment in Course 3
student_email = "neema.juma@student.com"
course_title = "Demo Course 3"

student = User.objects.filter(email=student_email).first()
course = Course.objects.filter(title=course_title).first()

if not student or not course:
    print(f"✗ Student ({student_email}) or Course ({course_title}) not found")
else:
    enrollment = Enrollment.objects.filter(student=student, course=course).first()
    if enrollment:
        print(f"✓ Found enrollment: {enrollment.id}")
        print(f"  Current Risk Score: {enrollment.risk_score}")
        print(f"  Current Risk Level: {enrollment.risk_level}")
        print(f"  Progress: {enrollment.progress_percentage}%")
        print(f"  Quiz Average: {enrollment.average_quiz_score}%")
        print(f"  Days Inactive: {enrollment.days_since_last_activity}")
    else:
        print(f"✗ No enrollment found")

## 2. View All Quiz Attempts

See the quiz history that contributes to risk calculation.

In [ ]:
# Get all quiz attempts for this enrollment
attempts = QuizAttempt.objects.filter(
    student=student,
    quiz__lesson__module__course=course,
    status=QuizAttempt.Status.COMPLETED
).order_by('-submitted_at')

print(f"Total Attempts: {attempts.count()}\n")

for i, attempt in enumerate(attempts, 1):
    age_days = (timezone.now() - attempt.submitted_at).days if attempt.submitted_at else 'N/A'
    status = "✓ PASS" if attempt.passed else "✗ FAIL"
    print(f"{i}. {attempt.quiz.title}")
    print(f"   Score: {attempt.score_percentage}% ({status})")
    print(f"   Date: {attempt.submitted_at} ({age_days} days ago)")
    print()

## 3. Calculate Risk with Mastery-Based Approach

Test the new risk calculation that weights recent performance more heavily.

In [ ]:
# Calculate risk using the mastery-based engine
risk_result = RiskEngine.calculate_student_risk(str(enrollment.id))

print("Risk Calculation Result:")
print(f"  Risk Score: {risk_result['risk_score']:.3f}")
print(f"  Risk Level: {risk_result['risk_level']}")
print(f"  Risk Trend: {risk_result['risk_trend']}")
print()
print("Components:")
for component, value in risk_result['components'].items():
    print(f"  {component}: {value}")
print()
print("Contributing Factors:")
for factor in risk_result['contributing_factors']:
    print(f"  - {factor['factor']} (impact: {factor['impact']})")

## 4. Simulate New Quiz Import

Simulate importing a new passing quiz score and recalculate risk.

In [ ]:
# Find or create a quiz for testing
quiz = Quiz.objects.filter(
    lesson__module__course=course
).first()

if not quiz:
    print("✗ No quiz found in course")
else:
    # Create a new passing attempt (simulating import)
    new_attempt = QuizAttempt.objects.create(
        student=student,
        quiz=quiz,
        attempt_number=QuizAttempt.objects.filter(student=student, quiz=quiz).count() + 1,
        status=QuizAttempt.Status.COMPLETED,
        score=85,
        score_percentage=85,
        max_possible_score=100,
        passed=True,
        started_at=timezone.now(),
        submitted_at=timezone.now(),
    )
    
    # Update enrollment performance
    new_attempt.update_enrollment_performance()
    
    # Recalculate risk
    risk_result = RiskEngine.calculate_student_risk(str(enrollment.id))
    
    print(f"✓ Created new attempt: 85% (PASS)")
    print(f"  New Quiz Average: {enrollment.average_quiz_score:.1f}%")
    print(f"  New Risk Score: {risk_result['risk_score']:.3f}")
    print(f"  New Risk Level: {risk_result['risk_level']}")
    
    # Check if lesson was auto-completed
    lesson_completed = LessonCompletion.objects.filter(
        student=student,
        lesson=quiz.lesson
    ).exists()
    print(f"  Lesson Auto-Completed: {lesson_completed}")

## 5. Check Alert Resolution

Verify that alerts are resolved when conditions improve.

In [ ]:
from analytics.models import Alert

# Check current alerts
alerts = Alert.objects.filter(enrollment=enrollment).order_by('-created_at')

print(f"Total Alerts: {alerts.count()}\n")

for alert in alerts[:5]:
    status_icon = "🔴" if alert.status == Alert.Status.ACTIVE else "✓" if alert.status == Alert.Status.RESOLVED else "⚠"
    print(f"{status_icon} {alert.alert_type} ({alert.status})")
    print(f"   Created: {alert.created_at.strftime('%Y-%m-%d')}")
    if alert.resolved_at:
        print(f"   Resolved: {alert.resolved_at.strftime('%Y-%m-%d')}")
        print(f"   Reason: {alert.intervention_outcome}")
    print()

## 6. Risk Score Breakdown

Visualize how the mastery-based calculation works.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calculate component contributions
progress_pct = float(enrollment.progress_percentage or 0)
quiz_avg_pct = float(enrollment.average_quiz_score or 0)
days_inactive = int(enrollment.days_since_last_activity or 0)

# Risk components (0-1, higher is worse)
progress_risk = 1.0 - (progress_pct / 100)
quiz_risk = 1.0 - (quiz_avg_pct / 100)
inactivity_risk = min(1.0, days_inactive / 14.0)

# Check for recent passes
thirty_days_ago = timezone.now() - timedelta(days=30)
has_recent_pass = QuizAttempt.objects.filter(
    student=student,
    quiz__lesson__module__course=course,
    submitted_at__gte=thirty_days_ago,
    passed=True
).exists()

# Adjust weights based on mastery
if has_recent_pass and quiz_avg_pct >= 60:
    weights = {'progress': 0.25, 'quiz': 0.50, 'inactivity': 0.25}
    weight_source = "Mastery-based (has recent pass)"
else:
    weights = {'progress': 0.40, 'quiz': 0.40, 'inactivity': 0.20}
    weight_source = "Standard"

# Calculate weighted risk
weighted_progress = progress_risk * weights['progress']
weighted_quiz = quiz_risk * weights['quiz']
weighted_inactivity = inactivity_risk * weights['inactivity']
total_risk = weighted_progress + weighted_quiz + weighted_inactivity

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Component values
categories = ['Progress', 'Quiz Avg', 'Inactivity']
raw_values = [progress_pct, quiz_avg_pct, 100 - (inactivity_risk * 100)]
risk_values = [progress_risk, quiz_risk, inactivity_risk]

ax1.barh(categories, raw_values, color=['#3b82f6', '#10b981', '#f59e0b'])
ax1.set_xlim(0, 100)
ax1.set_xlabel('Score / Percentage')
ax1.set_title('Raw Performance Metrics')
for i, v in enumerate(raw_values):
    ax1.text(v + 2, i, f'{v:.1f}%', va='center')

# Risk breakdown
risk_categories = ['Progress Risk', 'Quiz Risk', 'Inactivity Risk']
risk_contributions = [weighted_progress, weighted_quiz, weighted_inactivity]
colors = ['#ef4444' if r > 0.3 else '#f59e0b' if r > 0.15 else '#10b981' for r in risk_contributions]

ax2.barh(risk_categories, risk_contributions, color=colors)
ax2.set_xlim(0, 1)
ax2.set_xlabel('Risk Contribution (0-1)')
ax2.set_title(f'Risk Breakdown (Total: {total_risk:.3f})\n{weight_source}')
for i, v in enumerate(risk_contributions):
    ax2.text(v + 0.02, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.show()

print(f"\nWeight Method: {weight_source}")
print(f"Has Recent Pass: {has_recent_pass}")
print(f"Final Risk Score: {total_risk:.3f}")

## 7. Summary

Key improvements demonstrated:

1. **Mastery-Based Risk**: Recent passing scores reduce overall risk
2. **Alert Auto-Resolution**: Alerts resolved when conditions improve
3. **Lesson Auto-Completion**: Lessons marked complete when quizzes passed
4. **Dynamic Weighting**: Risk calculation adapts based on student performance

In [ ]:
# Final summary
print("═" * 60)
print("RISK CALCULATION TEST SUMMARY")
print("═" * 60)
print()
print(f"Student: {student.display_name}")
print(f"Course: {course.title}")
print()
print(f"Current Risk: {enrollment.risk_score:.3f} ({enrollment.risk_level})")
print(f"Quiz Average: {enrollment.average_quiz_score:.1f}%")
print(f"Progress: {enrollment.progress_percentage}%")
print()

active_alerts = Alert.objects.filter(
    enrollment=enrollment,
    status=Alert.Status.ACTIVE
).count()
resolved_alerts = Alert.objects.filter(
    enrollment=enrollment,
    status=Alert.Status.RESOLVED
).count()

print(f"Active Alerts: {active_alerts}")
print(f"Resolved Alerts: {resolved_alerts}")
print()
print("═" * 60)